The following has been adapted from Hailo's DFC Tutorials 1 and 2 (Parsing and Optimizing with DFC). It was run from a docker container setup using Hailo AI SoftwareSuite


In [ ]:
# General imports used throughout the tutorial
# file operations
import json
import os

import numpy as np
import tensorflow as tf
from IPython.display import SVG
from matplotlib import patches
from matplotlib import pyplot as plt
from PIL import Image
from tensorflow.python.eager.context import eager_mode

import torchvision as tv
import torch

import cv2

# import the hailo sdk client relevant classes
from hailo_sdk_client import ClientRunner, InferenceContext

%matplotlib inline

IMAGES_TO_VISUALIZE = 5

In [ ]:
chosen_hw_arch = "hailo8"

The ONNX model below was created using the `ultralytics` yolo11n.pt pretrained model, which was finetuned on a customised visdrone dataset (single class) and exported using:
```model.export(format='onnx', opset=14)```
    
the onnx output model was copied to the docker container into /local/shared_with_docker/yolov11n_visdrone.onnx
    
 

In [ ]:
onnx_model_name = "yolo11n_visdrone"
onnx_path = "/local/shared_with_docker/yolov11n_visdrone.onnx"

The endnodes below were taken from the [yolov11n.yaml](https://github.com/hailo-ai/hailo_model_zoo/blob/master/hailo_model_zoo/cfg/networks/yolov11n.yaml) on the hailo_model_zoo github. The finetuned (on VisDroneCustomClass) onnx model was opened in netron to double check these end nodes
```
- /model.23/cv2.0/cv2.0.2/Conv
- /model.23/cv3.0/cv3.0.2/Conv
- /model.23/cv2.1/cv2.1.2/Conv
- /model.23/cv3.1/cv3.1.2/Conv
- /model.23/cv2.2/cv2.2.2/Conv
- /model.23/cv3.2/cv3.2.2/Conv
```

In [ ]:
runner = ClientRunner(hw_arch=chosen_hw_arch)
hn, npz = runner.translate_onnx_model(
    onnx_path,
    onnx_model_name,
    start_node_names=["/model.0/conv/Conv"],
    end_node_names=["/model.23/cv2.0/cv2.0.2/Conv", 
                   "/model.23/cv3.0/cv3.0.2/Conv",
                   "/model.23/cv2.1/cv2.1.2/Conv",
                   "/model.23/cv3.1/cv3.1.2/Conv",
                   "/model.23/cv2.2/cv2.2.2/Conv",
                   "/model.23/cv3.2/cv3.2.2/Conv"],
    net_input_shapes={"/model.0/conv/Conv": [1, 3, 640, 640]},
)

In [ ]:
#save the parsed model
hailo_model_har_name = f"{onnx_model_name}_hailo_model_op14.har"
runner.save_har(hailo_model_har_name)

In [ ]:
!rm ./calib_set_visdrone.npy

In [ ]:
def preproc(image, output_height=640, output_width=640):
    preprocess = tv.transforms.Compose([
        tv.transforms.Resize((output_height, output_width)),
    ])
    
    data = np.array(preprocess(image))
    
    return data

In [ ]:
images_path = "../data/visdrone/visdrone/VisDrone2019-DET-train/images" # use training data for calib
images_list = [img_name for img_name in os.listdir(images_path) if os.path.splitext(img_name)[1] == ".jpg"]
dataset_sz = 3000 #len(images_list)
calib_dataset = np.zeros((dataset_sz, 640, 640, 3))

for idx, img_name in enumerate(sorted(images_list)):
    if idx==dataset_sz:
        break
    img = Image.open(os.path.join(images_path, img_name)).convert('RGB')
    img_preproc = preproc(img)
    calib_dataset[idx, :, :, :] = img_preproc


In [ ]:
#calib_dataset = np.load('calib_set_visdrone.npy')
calib_dataset.shape # check dataset shape; should be (1500, 640, 640, 3)

In [ ]:
import tensorflow as tf
import os
import numpy as np

# Paths to images and labels
images_dir = "../data/visdrone/visdrone/VisDrone2019-DET-train/images"
labels_dir = "..data/visdrone/visdrone/VisDrone2019-DET-train/labels"

# List of all images
image_files = sorted([os.path.join(images_dir, fname) for fname in os.listdir(images_dir) if fname.endswith('.jpg')])

# Create a function to read and process the image
def load_image(image_path):
    image = tf.io.read_file(image_path)  # Read image file
    image = tf.image.decode_jpeg(image, channels=3)  # Decode image to RGB format
    image = tf.image.resize(image, [640, 640])  # Resize image to a consistent size (e.g., 640x640)
    return image  # No normalization step here

# Create a function to parse the YOLO label files
def load_labels(label_path):
    label = tf.io.read_file(label_path)  # Read label file content as string
    label = tf.strings.strip(label)  # Strip any leading/trailing whitespaces
    
    # Split by newlines (each line corresponds to one object)
    labels = tf.strings.split(label, '\n')

    # Remove empty labels (if any)
    labels = [l for l in labels if tf.strings.length(l) > 0]

    # Parse each line in the format: class_id center_x center_y width height
    boxes = []
    for label in labels:
        parts = tf.strings.split(label, ' ')  # Split by space
        if len(parts) == 5:
            class_id = tf.strings.to_number(parts[0], tf.int32)
            center_x = tf.strings.to_number(parts[1], tf.float32)
            center_y = tf.strings.to_number(parts[2], tf.float32)
            width = tf.strings.to_number(parts[3], tf.float32)
            height = tf.strings.to_number(parts[4], tf.float32)
            boxes.append([class_id, center_x, center_y, width, height])
    
    boxes = tf.convert_to_tensor(boxes, dtype=tf.float32)
    return boxes

# Create a function to load and pair images with labels
def load_data(image_path):
    # Derive label path from image path (assuming the naming convention is consistent)
    label_path = tf.strings.regex_replace(image_path, 'images', 'labels')
    label_path = tf.strings.regex_replace(label_path, '.jpg$', '.txt')
    
    image = load_image(image_path)  # Load and preprocess the image (no normalization)
    labels = load_labels(label_path)  # Load and process labels
    
    return image, labels

# Create TensorFlow Dataset
dataset = tf.data.Dataset.from_tensor_slices(image_files)  # List of image file paths

# Use the map function with tf.data's `map` method to apply load_data to each image
dataset = dataset.map(lambda image_path: load_data(image_path), num_parallel_calls=tf.data.experimental.AUTOTUNE)

# Optional: shuffle, batch, and prefetch for performance
dataset = dataset.shuffle(buffer_size=1000)  # Shuffle dataset
dataset = dataset.batch(16)  # Set your batch size
dataset = dataset.prefetch(tf.data.experimental.AUTOTUNE)  # Prefetch to improve performance

# To verify the dataset works:
for img, lbl in dataset.take(1):
    print(f"Image shape: {img.shape}, Label shape: {lbl.shape}")


In [ ]:
#load our parsed HAR from the Parsing Tutorial
model_name = "yolo11n_visdrone"
hailo_model_har_name = "yolo11n_visdrone_hailo_model_op14.har"
assert os.path.isfile(hailo_model_har_name), "Please provide valid path for HAR file"
runner = ClientRunner(har=hailo_model_har_name)

The following was taken from trieut415's [guide for parsing/compiling a cuatom yolov11 model on colab](https://community.hailo.ai/t/guide-to-using-the-dfc-to-convert-a-modified-yolov11-on-google-colab/7131)

In [4]:
! ./YOLOv11nVisDroneCustomClassParsingOptimizingDFC.ipynb ../../../../../../../shared_with_docker/yolo11_end2end_parse_optimize_dfc.ipynb

/usr/bin/sh: 1: ./YOLOv11nVisDroneCustomClassParsingOptimizingDFC.ipynb: Permission denied


In [ ]:
from pprint import pprint

try:
    # Access the HailoNet as an OrderedDict
    hn_dict = runner.get_hn()  # Or use runner._hn if get_hn() is unavailable
    print("Inspecting layers from HailoNet (OrderedDict):")

    # Pretty-print each layer
    for key, value in hn_dict.items():
        print(f"Key: {key}")
        pprint(value)
        print("\n" + "="*80 + "\n")  # Add a separator between layers for clarity

except Exception as e:
    print(f"Error while inspecting hn_dict: {e}")

In [ ]:
# Now we will create a model script, that tells the compiler to add a normalization on the beginning
# of the model (that is why we didn't normalize the calibration set;
# Otherwise we would have to normalize it before using it)
alls =  """
normalization1 = normalization([0.0, 0.0, 0.0], [255.0, 255.0, 255.0])
change_output_activation(conv54, sigmoid)
change_output_activation(conv65, sigmoid)
change_output_activation(conv80, sigmoid)
nms_postprocess("../yolov11_nms_config_visdrone.json", meta_arch=yolov8, engine=cpu)
allocator_param(width_splitter_defuse=disabled)

model_optimization_config(calibration, batch_size=16, calibset_size=512)
post_quantization_optimization(finetune, policy=enabled, learning_rate=0.0001, epochs=8, dataset_size=2400)
 """

# Load the model script to ClientRunner so it will be considered on optimization
runner.load_model_script(alls)

In [ ]:
# load the Quantized HAR file
model_name = "yolo11n_visdrone"
quantized_model_har_path = f"{model_name}_quantized_model_visdrone.har"
runner = ClientRunner(har=quantized_model_har_path, hw_arch=chosen_hw_arch)


In [ ]:
# Call Optimize to perform the optimization process
runner.optimize(calib_dataset)

# Save the result state to a Quantized HAR file
quantized_model_har_path = f"{model_name}_quantized_model_visdrone.har"
runner.save_har(quantized_model_har_path)

In [ ]:
sample_dataset = np.zeros((2, 640, 640, 3))
SAMPLE_IMAGE_PATH = '../data/visdrone/val/images/0000364_01569_d_0000781.jpg'
img = Image.open(SAMPLE_IMAGE_PATH).convert('RGB')
img_preproc = preproc(img)
sample_dataset[0,:,:,:] = img_preproc

# Notice that we use the original images, because normalization is IN the model
with runner.infer_context(InferenceContext.SDK_QUANTIZED) as ctx:
    output = runner.infer(ctx, sample_dataset[:1, :, :, :])

In [ ]:
output.shape

In [ ]:
# remove all non-zero padding elements
valid_mask = np.any(output != 0, axis=(1, 2))  # Check if any non-zero values exist in each column
last_valid_indices = np.argmax(~valid_mask, axis=1) # First occurrence of zero padding
last_valid_indices[valid_mask[:, -1]] = output.shape[-1] # edge case, no detections
dets = output[0, 0, :, :last_valid_indices[0]]

dets = dets.transpose()
dets

In [ ]:
imgsz = 640
detsmul = [x * imgsz for x in dets]

exp_labels= []
for el in detsmul:
    exp_labels.append([el[1], el[0], el[3], el[2]])          


In [ ]:
# Load the image
SAMPLE_IMAGE_PATH = "../data/visdrone/train/images/0000002_00005_d_0000014.jpg"
imgpth = SAMPLE_IMAGE_PATH
img = cv2.imread(imgpth)

labpth = imgpth.replace("images", "labels").replace('jpg', 'txt')

# Get image dimensions
height, width, _ = img.shape
# Scaling factors
scale_x = width / 640
scale_y = height / 640

preds = []
for idx, el in enumerate(exp_labels):
# Rescale the coordinates
    x1_orig = int(el[0] * scale_x)
    y1_orig = int(el[1] * scale_y)
    x2_orig = int(el[2] * scale_x)
    y2_orig = int(el[3] * scale_y)
    preds.append([0, x1_orig, y1_orig, x2_orig, y2_orig])

In [ ]:
#Ground truth labels in YOLO format (class_id, x_center, y_center, bbox_width, bbox_height)
with open(labpth, "r") as f:
    gtlabels = [tuple(map(float, line.split())) for line in f]
    f.close()
    
print(gtlabels)
    


# Loop through labels and draw bounding boxes
for idx, labels in enumerate(gtlabels):
    class_id, x_center, y_center, bbox_width, bbox_height = labels

    # Convert normalized YOLO coordinates to pixel values
    x_center, y_center = int(x_center * width), int(y_center * height)
    bbox_width, bbox_height = int(bbox_width * width), int(bbox_height * height)

    # Calculate top-left and bottom-right corners
    x1_gt = int(x_center - bbox_width / 2)
    y1_gt = int(y_center - bbox_height / 2)
    x2_gt = int(x_center + bbox_width / 2)
    y2_gt = int(y_center + bbox_height / 2)
    print(f"ground truth: {[x1_gt, y1_gt, x2_gt, y2_gt]}")
    
    # Draw the bounding box
    colorgt = (0, 255, 0)  # Green color for bounding box
    colorprd = (150, 50, 100)  # Green color for bounding box

    thickness = 2
    cv2.rectangle(img, (x1_gt, y1_gt), (x2_gt, y2_gt), colorgt, thickness)

    # Add class label text
    cv2.putText(img, str(class_id), (x1_gt, y1_gt - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, colorgt, 2)

    if idx < len(preds):
        print(f"predictions: {preds[idx][1:]}")
        clsid, x1_prd, y1_prd, x2_prd, y2_prd = preds[idx]
        cv2.rectangle(img, (x1_prd, y1_prd), (x2_prd, y2_prd), colorprd, thickness)

    


In [ ]:
# Display the image with bounding boxes
cv2.imshow("Ground Truth", img)
cv2.waitKey(0)  # Wait for key press
cv2.destroyAllWindows()  # Close window

In [ ]:

import cv2
import numpy as np

# Load the image
SAMPLE_IMAGE_PATH = '/media/citi-ai/matthew/uav-human-detection/hailo-ai/shared_with_docker/0000002_00005_d_0000014.jpg'
img = cv2.imread(SAMPLE_IMAGE_PATH)

# Get image dimensions
height, width, _ = img.shape
# Scaling factors
scale_x = width / 640
scale_y = height / 640

preds = []
for idx, el in enumerate(exp_labels):
# Rescale the coordinates
    x1_orig = int(el[0] * scale_x)
    y1_orig = int(el[1] * scale_y)
    x2_orig = int(el[2] * scale_x)
    y2_orig = int(el[3] * scale_y)
    print(f"Index: {idx}:\n{x1_orig}, {y1_orig}, {x2_orig}, {y2_orig}")
    preds.append([62, x1_orig, y1_orig, x2_orig, y2_orig])


# Ground truth labels in YOLO format (class_id, x_center, y_center, bbox_width, bbox_height)
gtlabels = [(0, 0.434896, 0.391667, 0.005208, 0.016667),
            (0, 0.450000, 0.376852, 0.008333, 0.024074),
            (0, 0.921875, 0.491667, 0.012500, 0.035185),
            (0, 0.938542, 0.432407, 0.008333, 0.031481),
            (0, 0.062500, 0.598148, 0.010417, 0.037037),
            (0, 0.245312, 0.450000, 0.007292, 0.025926),
            (0, 0.250000, 0.450000, 0.006250, 0.025926),
            (0, 0.066146, 0.402778, 0.009375, 0.024074),
            (0, 0.093750, 0.375000, 0.008333, 0.031481),
            (0, 0.231771, 0.227778, 0.007292, 0.018519),
            (0, 0.067187, 0.535185, 0.011458, 0.033333),
]


# Loop through labels and draw bounding boxes
for idx, labels in enumerate(gtlabels):
    class_id, x_center, y_center, bbox_width, bbox_height = labels

    # Convert normalized YOLO coordinates to pixel values
    x_center, y_center = int(x_center * width), int(y_center * height)
    bbox_width, bbox_height = int(bbox_width * width), int(bbox_height * height)

    # Calculate top-left and bottom-right corners
    x1_gt = int(x_center - bbox_width / 2)
    y1_gt = int(y_center - bbox_height / 2)
    x2_gt = int(x_center + bbox_width / 2)
    y2_gt = int(y_center + bbox_height / 2)
    print(f"ground truth: {[x1_gt, y1_gt, x2_gt, y2_gt]}")
    print(f"predictions: {preds[idx][1:]}")

    clsid, x1_prd, y1_prd, x2_prd, y2_prd = preds[idx]

    # Draw the bounding box
    colorgt = (0, 255, 0)  # Green color for bounding box
    colorprd = (150, 50, 100)  # Green color for bounding box

    thickness = 2
    cv2.rectangle(img, (x1_gt, y1_gt), (x2_gt, y2_gt), colorgt, thickness)
    cv2.rectangle(img, (x1_prd, y1_prd), (x2_prd, y2_prd), colorprd, thickness)

    # Add class label text
    cv2.putText(img, str(class_id), (x1_gt, y1_gt - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, colorgt, 2)


In [ ]:
# Display the image with bounding boxes
# cv2.imshow("Ground Truth", img)
# cv2.waitKey(0)  # Wait for key press
# cv2.destroyAllWindows()  # Close window